# Show3D - real Samsung logic_013

Native Samsung ptychography stacks from SSD. The widgets below force `offline=False` so the notebook uses float32 data instead of packed uint8 report data.

In [1]:
%env ANYWIDGET_HMR=0

from pathlib import Path
import json
import numpy as np
from IPython.display import display
from quantem.widget import Show3D

env: ANYWIDGET_HMR=0


/home/owner/miniforge3/envs/cuda-env/lib/python3.13/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
TRIAL = Path("/home/owner/ssd/data/samsung/20260408_logic_device/quantem/ptycho/logic_013/trials/190_det192_scan0_s16_t20_p12_it10_pure_phase_decay")
GAAFET = Path("/home/owner/reports/20260513_samsung_gaafet_ptycho")

cfg = json.loads((TRIAL / "config.json").read_text())
phase = np.load(TRIAL / "obj_phase.npy").astype(np.float32, copy=False)
sampling_A = float(cfg["data"]["scan_sampling_A"])

print(f"logic_013 phase: {phase.shape}, {phase.dtype}, {phase.nbytes / 1024**2:.1f} MiB")
print(f"sampling: {sampling_A:.4f} A/px")

logic_013 phase: (16, 1688, 1688), float32, 173.9 MiB
sampling: 0.4949 A/px


## Single full-resolution Samsung stack

In [3]:
single = Show3D(
    phase,
    panel_titles=["logic_013 trial190 full"],
    title="Samsung logic_013 trial190 - native full frame",
    cmap="magma",
    pixel_size=sampling_A,
    pixel_unit="A",
    dim_label="slice",
    fps=120,
    offline=False,
    show_fft=False,
    smooth=False,
    size=760,
)
display(single)

Show3D(16×1688×1688, frame=8, cmap=magma)

## Nine real Samsung panels

Five panels are different native crops from the full logic_013 stack; four panels are real cropped Samsung GAAFET/logic reconstructions. No synthetic data, no binning.

In [4]:
def crop_stack(stack, top, left, size=1370):
    return np.ascontiguousarray(stack[:, top:top + size, left:left + size], dtype=np.float32)

crop_size = 1370
h, w = phase.shape[1:]
max_top = h - crop_size
max_left = w - crop_size
center = (max_top // 2, max_left // 2)

panels = [
    crop_stack(phase, center[0], center[1], crop_size),
    crop_stack(phase, 0, 0, crop_size),
    crop_stack(phase, 0, max_left, crop_size),
    crop_stack(phase, max_top, 0, crop_size),
    crop_stack(phase, max_top, max_left, crop_size),
]
labels = [
    "logic center",
    "logic top-left",
    "logic top-right",
    "logic bottom-left",
    "logic bottom-right",
]

for folder, label in [
    ("trial_076_iter10", "GAAFET iter10"),
    ("trial_080_iter50", "GAAFET iter50"),
    ("trial_055_iter100", "GAAFET iter100"),
    ("trial_190_iter10_logic013", "logic cropped"),
]:
    panels.append(np.load(GAAFET / folder / "obj_phase_cropped.npy").astype(np.float32, copy=False))
    labels.append(label)

print(f"panels: {len(panels)} x {panels[0].shape}, total source {sum(p.nbytes for p in panels) / 1024**2:.1f} MiB")

panels: 9 x (16, 1370, 1370), total source 1031.0 MiB


In [5]:
samsung_9panel = Show3D(
    *panels,
    panel_titles=labels,
    title="Samsung real-data 9-panel scrub - float32 live path",
    cmap="magma",
    pixel_size=sampling_A,
    pixel_unit="A",
    dim_label="slice",
    fps=120,
    offline=False,
    show_fft=False,
    show_stats=False,
    smooth=False,
    max_cols=3,
    panel_gap=8,
    panel_title_font_size=10,
    link_contrast=True,
    link_panels=True,
    size=360,
)
display(samsung_9panel)

Show3D(16×1370×12330, frame=8, cmap=magma)